In [1]:
import pandas as pd
from datasets import load_dataset
import re

print("--- Iniciando proceso de creación del Dataset Maestro ---")

# 1. DESCARGAR DATASET NEUTRO DESDE HUGGING FACE
print("Descargando dataset de frases neutras (jaimevera1107)...")
dataset_hf = load_dataset("jaimevera1107/similarity-sentences-spanish")
df_hf = pd.DataFrame(dataset_hf['train'])

# Aplanamos las frases de 'sentence1' y 'sentence2'
todas_neutras = df_hf['sentence1'].tolist() + df_hf['sentence2'].tolist()
df_neutro_total = pd.DataFrame({'texto': todas_neutras, 'toxicidad': 0}).drop_duplicates()

# 2. CARGAR TU DATASET DE TÓXICOS (el que ya limpiamos antes)
# Asegúrate de que el archivo 'train_dataset_cleaned.csv' esté en la misma carpeta
print("Cargando tus mensajes tóxicos y lista de insultos...")
df_toxic_completo = pd.read_csv('train_dataset_cleaned.csv')

# 3. SEPARAR CLASES PARA BALANCEO (Unidad 1)
df_solo_toxicos = df_toxic_completo[df_toxic_completo['toxicidad'] == 1]
df_solo_neutros = df_neutro_total # Nuestras nuevas frases de Hugging Face

# Determinamos el número para que sea 50/50
# Usaremos el número de frases neutras disponibles para no repetir datos
n_muestras = min(len(df_solo_toxicos), len(df_solo_neutros))

print(f"Balanceando datos: seleccionando {n_muestras} ejemplos de cada clase...")

df_tox_bal = df_solo_toxicos.sample(n=n_muestras, random_state=42)
df_neu_bal = df_solo_neutros.sample(n=n_muestras, random_state=42)

# 4. UNIÓN Y MEZCLA FINAL
df_maestro = pd.concat([df_tox_bal, df_neu_bal], ignore_index=True)
df_maestro = df_maestro.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. GUARDAR RESULTADO
nombre_archivo = 'DATASET_MAESTRO_BALANCEADO.csv'
df_maestro.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')

print(f"\n--- PROCESO COMPLETADO ---")
print(f"Archivo guardado como: {nombre_archivo}")
print(f"Total de filas: {len(df_maestro)}")
print(f"Distribución:\n{df_maestro['toxicidad'].value_counts()}")
print("\nPrimeras 5 filas:")
print(df_maestro.head())

c:\Users\joanv\anaconda3\envs\moderacion_ia\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Iniciando proceso de creación del Dataset Maestro ---
Descargando dataset de frases neutras (jaimevera1107)...


c:\Users\joanv\anaconda3\envs\moderacion_ia\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joanv\.cache\huggingface\hub\datasets--jaimevera1107--similarity-sentences-spanish. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 17532/17532 [00:00<00:00, 125766.71 exa

Cargando tus mensajes tóxicos y lista de insultos...
Balanceando datos: seleccionando 27747 ejemplos de cada clase...

--- PROCESO COMPLETADO ---
Archivo guardado como: DATASET_MAESTRO_BALANCEADO.csv
Total de filas: 55494
Distribución:
toxicidad
0    27747
1    27747
Name: count, dtype: int64

Primeras 5 filas:
                                               texto  toxicidad
0  Kathy Nicolo (Jennifer Connelly) está desesper...          0
1               Un cachorro corriendo por una calle.          0
2  Te juro que si vuelves a contradecirme, te arr...          1
3  Mis palabras son armas, mi voz es el terror y ...          1
4  Los musulmanes y su profeta Mahoma son una ver...          1
